# Readme

This document contains exercies related to parametric models: **Ridge regression**, **Polynomial dependence models**. 
Goal is to build intuition behind overfitting and underfitting of models caused by choosed parameter.  

In [ ]:
# basic imports 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
# transformators 
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
# models 
from sklearn.linear_model import LinearRegression, Ridge
# model selection
from sklearn.model_selection import cross_val_score

# '--------------------------------------------'

# Task 1

Create a data (with n=50 datapoints) according to the formula $y=1+2x^2+e$, where $x\in [-2,3]$ and $e\sim \mathcal{N}(0,1)$. Fit the model of the form (use `PolynomialFeatures`):
- $y=\beta_0+\beta_1x$
- $y=\beta_0+\beta_1x+\beta_2x^2$
- $y=\beta_0+\beta_1x+\beta_2x^2+\beta_3 x^3+ \ldots +\beta_{20} x^{20}$

Prepare the plots with true and fitted models.



### Solution

##### **`Goal`**
We want to see how degree of polynomial affects how model is fitted

##### **`Method`**
We iterated through a set of predefined polynomial degrees. For each degree:  
- a model was created using polynomial features of the corresponding degree,  
- the model was trained on the available data,  
- and the resulting fit was visualized to assess model behavior.  

##### **`Results and Interpretation`**  
This experiment clearly illustrates the concepts of **underfitting** and **overfitting**.  
- When the polynomial degree is too **low**, the model is too simple to capture the underlying pattern in the data. As a result, predictions are inaccurate — this is a case of **underfitting**.  
- When the degree is too **high**, the model becomes overly complex and starts fitting not only the true signal but also the noise in the data. This leads to poor generalization and unstable predictions — a example of **overfitting**.

The exercise effectively demonstrates how model complexity impacts performance and highlights the importance of selecting an appropriate polynomial degree to balance bias and variance.





In [ ]:
# parameters 
n = 200

# creating data
X_dom = np.linspace(-2, 3, 100)
X = np.random.uniform(-2, 3, n).reshape(-1, 1)
e = np.random.normal(size=n)
y = (1 + 2*X.reshape(-1, )**2 + e ).reshape(-1, 1)

# creating polynomial models 
for i in (1, 2, 20):
    # training model 
    Poly = PolynomialFeatures(i)
    train_data = Poly.fit_transform(X.reshape(-1, 1))
    model = LinearRegression()
    model.fit(train_data, y)
    # plot values
    X_test_domain = Poly.transform(X_dom.reshape(-1, 1))
    val = model.predict(X_test_domain)
    # plot
    plt.plot(X_dom, val, color='red')
    plt.scatter(X, y)
    plt.ylim(-1 + y.min(), 1 + y.max())
    plt.show()
    print(model.score(train_data, y))


# Task 2

Using the data from Task 1, find the best polynomial degree using cross-validation. Visualise the results.

### Solution

##### **`Goal`**
The aim is to evaluate how accurately the prediction from cross-validation approximates the true value of the polynomial.

##### **`Methods`**
we calculate error scoring and we choose polynomial rank for which the error is smallest. 
- first cell contains plot of one iteration of **mean squared error**
- second cell contains plot of one iteration of **mean absolute error**
- thid cell contains code which calucalte mean of choosen degrees for specified probe size using **mean squared error**. 10 samples per size $n$ 

##### **`Results & discussion`**
Cross-validation generally returns either the correct or a higher polynomial degree. The experiment was designed to investigate how the sample size affects the selected polynomial degree.

Although the number of repetitions per sample size is relatively low (10), the results suggest that the most stable behavior occurs with relatively small samples. With larger samples overfitted models are preferred. 

This could be due to the fact that as the sample size increases, the contribution of variance decreases, and cross-validation tends to favor more complex models (see pages 197, 208 in *The Elements of Statistical Learning*).

In [ ]:
# original task 
n = 50
X_dom = np.linspace(-2, 3, 100)
X = np.random.uniform(-2, 3, n).reshape(-1, 1)
e = np.random.normal(size=n)
y = (1 + 2*X.reshape(-1, )**2 + e ).reshape(-1, 1)

# parameters
degrees = range(1, 21)
scores_values = []

for d in degrees:
    # creating model
    model = make_pipeline(
        PolynomialFeatures(d),
        LinearRegression()
    )
    # cross validation 
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error') # TODO scoring 
    scores_values.append(-scores.mean())

# plot 
plt.figure(figsize=(10, 5))
plt.plot(degrees, scores_values, marker='o')
plt.xlabel("Polynomial degree")
plt.ylabel("Cross-validated MSE")
plt.title("Cross-validation error vs. polynomial degree")
plt.grid(True)
plt.show()

degrees[np.argmin(scores_values)], np.min(scores_values)

In [ ]:

# parameters
degrees = range(1, 21)
scores_values = []

for d in degrees:
    # creating model
    model = make_pipeline(
        PolynomialFeatures(d),
        LinearRegression()
    )
    # cross validation 
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error') # TODO scoring 
    scores_values.append(-scores.mean())

# plot 
plt.figure(figsize=(10, 5))
plt.plot(degrees, scores_values, marker='o')
plt.xlabel("Polynomial degree")
plt.ylabel("Cross-validated MSE")
plt.title("Cross-validation error vs. polynomial degree")
plt.grid(True)
plt.show()

degrees[np.argmin(scores_values)], np.min(scores_values)

In [ ]:

# version with finding mean optimal degree for probe size

# index 
n_list = [50*i for i in range(1, 11)]

# calculation 
all_scores = []
for n in n_list:
    # creating data
    mean_score = []
    for i in range(10):
        X_dom = np.linspace(-2, 3, 100)
        X = np.random.uniform(-2, 3, n).reshape(-1, 1)
        e = np.random.normal(size=n)
        y = (1 + 2*X.reshape(-1, )**2 + e ).reshape(-1, 1)

        # parameters
        degrees = range(1, 21)
        scores_values = []

        for d in degrees:
            # creating model
            model = make_pipeline(
                PolynomialFeatures(d),
                LinearRegression()
            )
            # cross validation 
            scores = cross_val_score(model, X, y, cv=5,scoring='neg_mean_squared_error') # TODO scoring 
            scores_values.append(scores.mean())
        best_degree = degrees[np.argmax(scores_values)]
        mean_score.append(best_degree)
    all_scores.append(np.mean(mean_score))

    # # plot 
    # plt.figure(figsize=(10, 5))
    # plt.plot(degrees, scores_values, marker='o')
    # plt.xlabel("Polynomial degree")
    # plt.ylabel("Cross-validated MSE")
    # plt.title("Cross-validation error vs. polynomial degree")
    # plt.grid(True)
    # plt.show()

    # best degree
    best_degree = degrees[np.argmax(scores_values)]
    # print(f"Best polynomial degree (based on CV): {best_degree} with $R^2$ score $={np.max(scores_values)}$")
pd.DataFrame(np.array(all_scores), index=n_list, columns=['Degree'])

# Task 3.
Use the above dataset. Fit polynomial ridge regression (degree 20) with lambda equal 1 and 100 (in the penalty function take into account all coeaficionts). Use scaling.

Draw the data and all curves on one figure.


Compare the new curves on the graph with a polynomial regression of degree 20.


### Solution

#### Method  
The method is the same as in the previous exercise.  
This time, however, we fix the polynomial degree at **20** and apply **Ridge regression** with different regularization strengths:  
- λ = 1  
- λ = 100  

#### Results and Interpretation  
We observe how Ridge regression influences the model by **reducing the variance of the coefficients**, even though the model itself remains complex.  
- The regularization term helps to **stabilize the model**, making the coefficients smaller and more balanced.  
- As a result, although the model still exhibits some overfitting due to high polynomial degree, it becomes **more reliable for prediction**.
- $\lambda=100$ wants to smooth coefficents to much and model is not well fitted 

$R^2_{\mathrm{\lambda = 1}} = 0.96 > 0.76 = R^2_{\mathrm{\lambda = 100}}$

This experiment demonstrates how Ridge regression can mitigate the negative effects of overfitting by controlling coefficient magnitudes, thus improving generalization.


In [ ]:
# parameters
deg = 20 

# exercise 
for a in (1, 100):
    # creating model
    model = make_pipeline(
        PolynomialFeatures(deg),
        StandardScaler(),
        Ridge(a)
        )

    # training model
    model.fit(X.reshape(-1, 1), y)
    # creating plot
    pred = model.predict(X_dom.reshape(-1, 1))
    plt.plot(X_dom, pred, label=f'Model : $\lambda = {a}$')
    print(model.score(X, y))

plt.scatter(X, y)
plt.ylim(y.min() - 1, y.max() + 1)
plt.legend()
plt.show()

# printing plot 


# Task 4.
In Task 3 find the best parameter C by using cross-validation.

Compare the new curves on the graph with a polynomial regression of degree 20.


### Solution 

###### **`Goal`**
we want to optimize last problem so we can see most optimal $\lambda$ parameter

##### **`Method`**
we create discrete number of arguments. We iterate through each and calculate mean value squared error which we later compare and choose minimal value from it. 

##### **`Results & discussion`**
Optimal parameter is $\lambda = 0.908$. We can see that minimal values are close to $1$. With $\lambda$ growing our error is rising. This can be due too increasing biased error of model. 


In [ ]:

# parameters
deg = 20 
c_domain = np.linspace(0.1, 10, 50)
scores_values = []

# exercise 
for c in c_domain:
    # creating model
    model = make_pipeline(
        PolynomialFeatures(deg),
        StandardScaler(),
        Ridge(c)
        )

    # training model
    model.fit(X, y)
    # finding score
    pred = model.predict(X_dom.reshape(-1, 1))
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
    scores_values.append(-scores.mean())
# choosing best parameter
best_c = c_domain[np.argmin(scores)]
print(best_c)
print(scores_values)

plt.plot(c_domain, scores_values)
plt.ylim(min(scores_values) - 0.2, max(scores_values) + 0.2)
plt.legend()
plt.show()

# printing plot 


# Task 5
Calculate (manually) Rigde regression beta parameters for 
$\lambda=1$,
$x_1=[0,1,2,3]$,
$x_2=[0,0,1,1]$,
$y=[2,2,3]$.

#### Data:
Dimension of $y$ is $3$ and dimension of $x$ is $4$. In order to make dimension similiar i am dropping values from features on last index. 


#### Method
To find Ridge regression parameters we need to optimilize cost function of form:
 
$(y - X\beta)^t(y - X\beta) + \lambda \beta^t \beta$

After solving minimization problem we obtain that our estimator should be of form: 

$\hat\beta = \left(X^tX + \lambda I\right)^{-1} X^ty$

To calculate $\hat\beta$ including intercept we adding ones to our feature column $X$, and we adjusting $I$ so $I_{0, 0} = 0$. It is because adding weight penalty for $\beta_0$ would make procedure dependent on origin. (The elements of statistical learning chapter 3 page 59) 


#### Solution

We can see that manually calculating ridge regression gave ous same result as using built in method fit on object Ridge from sklearn.linear_model


In [ ]:
# creating X, y variables
c = 1
X_1 = np.array([
    [0, 1, 2],
    [0, 0, 1]
]).reshape(3, 2)
# adding intercept 
X = np.concat((np.ones((X_1.shape[0], 1)), X_1), axis=1)
y = np.array([2, 2, 3]).reshape(-1, 1)

# finding b parameter WARNING we need to not evaluate on eye our identity object 
I = np.eye(X.shape[1]) 
I[0, 0] = 0
b = np.linalg.solve((X.T @ X + c * I), X.T @ y)

# finding b from build in regression 
lm = Ridge(c, fit_intercept=True)
lm.fit(X_1, y)
print(np.concat((lm.intercept_, lm.coef_)), '\n', b.reshape(-1, ))


